# 09 · A small rule framework, then Great Expectations

First understand the Spark expressions; then reuse them. Great Expectations is optional and isolated from the core lessons. The core framework below deliberately supports only two rule types so students can read every line.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Declarative rules without a large framework
Separate rule configuration from expression construction. Missing values explicitly fail. Unsupported rule types raise instead of silently skipping a contract.


In [ ]:
rule_specs = [
    dict(rule_id="DQ001", column="customer_id", type="not_blank"),
    dict(rule_id="DQ002", column="quantity", type="range", min=1, max=1000),
]


def compile_rule(spec):
    c = F.col(spec["column"])
    if spec["type"] == "not_blank":
        return c.isNotNull() & (F.length(F.trim(c)) > 0)
    if spec["type"] == "range":
        return F.expr(f"try_cast(`{spec['column']}` as decimal(18,2))").between(
            spec["min"], spec["max"]
        )
    raise ValueError("Unsupported rule type: " + spec["type"])


demo = spark.createDataFrame(
    [("C001", "2"), (None, "0"), (" ", "two")], "customer_id string, quantity string"
)
for spec in rule_specs:
    demo = demo.withColumn(
        spec["rule_id"], F.coalesce(compile_rule(spec), F.lit(False))
    )
demo.filter("DQ001 and DQ002").show()
demo.filter("not (DQ001 and DQ002)").show()


## Optional GX Core 1.x setup
Install `great_expectations>=1.3,<2` into a separate compatible Spark kernel using `requirements-gx.txt`, restart it, then set RUN_GX=True. Do not replace the platform-provided PySpark package on EMR/Glue. GX is skipped by default so the core lab has no dependency. The API below uses the documented Core 1.x Spark DataFrame workflow, not the legacy 0.18 API.


In [ ]:
RUN_GX = False
if RUN_GX:
    import great_expectations as gx

    print("GX runtime version:", gx.__version__)
    context = gx.get_context(mode="ephemeral")
    source = context.data_sources.add_spark(name="orders_source_" + RUN_ID)
    asset = source.add_dataframe_asset(name="orders")
    definition = asset.add_batch_definition_whole_dataframe(name="whole_batch")
    gx_df = spark.createDataFrame(
        [("O1", "C1", 2, "PAID"), ("O1", None, -1, "OTHER")],
        "order_id string, customer_id string, quantity int, status string",
    )
    batch = definition.get_batch(batch_parameters={"dataframe": gx_df})
    expectations = [
        gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"),
        gx.expectations.ExpectColumnValuesToBeUnique(column="order_id"),
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="quantity", min_value=1, max_value=1000
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="status", value_set=["CREATED", "PAID", "SHIPPED", "CANCELLED"]
        ),
        gx.expectations.ExpectTableColumnsToMatchOrderedList(
            column_list=["order_id", "customer_id", "quantity", "status"]
        ),
    ]
    for expectation in expectations:
        result = batch.validate(expectation)
        print(expectation.__class__.__name__, result.success, result.result)
else:
    print("Optional GX extension skipped. Manual Spark rules above were executed.")


## What changes with a framework?
Spark filters become reusable Expectations; suites collect contracts; validation results support shared reporting and checkpoints. These do not automatically implement source accounting, quarantine replay, or your publication policy. Keep those decisions explicit.

| Manual check | GX expectation |
|---|---|
| isNotNull | ExpectColumnValuesToNotBeNull |
| duplicate key count | ExpectColumnValuesToBeUnique |
| between | ExpectColumnValuesToBeBetween |
| isin | ExpectColumnValuesToBeInSet |
| ordered schema columns | ExpectTableColumnsToMatchOrderedList |

[Official Spark DataFrame connection guide](https://docs.greatexpectations.io/docs/core/connect_to_data/dataframes/).
